# Notebook 02 — Query Embedding & Retrieval Observability

This notebook observes the **Phase 2 retrieval pipeline** by hitting the live FastAPI endpoints.

What we will inspect:
1. Embed a query and examine its vector
2. Compare vectors for different queries
3. Retrieve top-K passages for several queries and inspect ranked results
4. Visualise similarity scores

**Prerequisite:** the API server must be running and data must be ingested.
```
uv run uvicorn main:app --app-dir src --port 8004
# POST http://localhost:8004/ingest?limit=50
```

In [1]:
import math
import requests
import pandas as pd
from IPython.display import display

BASE_URL = "http://localhost:8004"

resp = requests.get(f"{BASE_URL}/health")
print("Server:", resp.json()["status"])

Server: ok


## 1. Embed a Query

Call `POST /embed` to turn a text query into a vector and inspect its properties.

In [2]:
query = "What is the capital of Uruguay?"
resp = requests.post(f"{BASE_URL}/embed", params={"query": query})
body = resp.json()

vec = body["embedding"]
magnitude = math.sqrt(sum(v**2 for v in vec))

print(f"Query         : {body['query']}")
print(f"Dimensions    : {body['dimensions']}")
print(f"Vector norm   : {magnitude:.6f}")
print(f"First 8 values: {[round(v, 6) for v in vec[:8]]}")

Query         : What is the capital of Uruguay?
Dimensions    : 3072
Vector norm   : 1.000000
First 8 values: [0.005122, 0.072526, -0.004127, 0.004006, -0.028021, 0.024297, -0.022038, -0.004931]


## 2. Compare Embeddings for Different Queries

Different queries should produce different vectors. Related queries should be closer in vector space.

In [3]:
def cosine_similarity(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    norm1 = math.sqrt(sum(a**2 for a in v1))
    norm2 = math.sqrt(sum(b**2 for b in v2))
    return dot / (norm1 * norm2)

queries = [
    "What is the capital of Uruguay?",
    "Where is Montevideo located?",      # semantically close to above
    "How many legs does a spider have?", # semantically distant
]

embeddings = {}
for q in queries:
    r = requests.post(f"{BASE_URL}/embed", params={"query": q})
    embeddings[q] = r.json()["embedding"]

# Print pairwise cosine similarities
print("Pairwise cosine similarities:\n")
for i, q1 in enumerate(queries):
    for j, q2 in enumerate(queries):
        if j > i:
            sim = cosine_similarity(embeddings[q1], embeddings[q2])
            print(f"  '{q1[:40]}' vs")
            print(f"  '{q2[:40]}'")
            print(f"  → similarity = {sim:.4f}\n")

Pairwise cosine similarities:

  'What is the capital of Uruguay?' vs
  'Where is Montevideo located?'
  → similarity = 0.6232

  'What is the capital of Uruguay?' vs
  'How many legs does a spider have?'
  → similarity = 0.0390

  'Where is Montevideo located?' vs
  'How many legs does a spider have?'
  → similarity = 0.0466



## 3. Retrieve Top-K Passages

Call `POST /retrieve` and inspect the ranked results — rank, score, passage text, and source ID.

In [4]:
query = "What is Uruguay?"
resp = requests.post(f"{BASE_URL}/retrieve", params={"query": query, "top_k": 5})
body = resp.json()

print(f"Query  : {body['query']}")
print(f"Top-K  : {body['top_k']}")
print(f"Results: {len(body['results'])}\n")

for r in body["results"]:
    print(f"  Rank {r['rank']}  score={r['score']:.4f}  passage_id={r['passage_id']}")
    print(f"  {r['text'][:120]}...")
    print()

Query  : What is Uruguay?
Top-K  : 5
Results: 5

  Rank 1  score=0.6150  passage_id=0
  Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part o...

  Rank 2  score=0.5709  passage_id=25
  At 176,214 square kilometres (68,036 square miles) of continental land and 142,199 square kilometres (54,903 sq mi) of j...

  Rank 3  score=0.5657  passage_id=37
  Uruguay has a middle income economy, mainly dominated by the State services sector, an export-oriented agricultural sect...

  Rank 4  score=0.5468  passage_id=14
  As a province of the Viceroyalty of La Plata, colonial Uruguay was known as the Banda Oriental, or Eastern Strip, referr...

  Rank 5  score=0.5439  passage_id=18
  Uruguay's politics takes place in a framework of a presidential representative democratic republic, whereby the Presiden...



In [5]:
# Display results as a DataFrame
rows = [
    {"rank": r["rank"], "score": round(r["score"], 4), "passage_id": r["passage_id"],
     "text_preview": r["text"][:80] + "..."}
    for r in body["results"]
]
display(pd.DataFrame(rows))

,rank,score,passage_id,text_preview
0,1,0.6150,0,"Uruguay (official full name in ; pron. , Eas..."
1,2,0.5709,25,"At 176,214 square kilometres (68,036 square mi..."
2,3,0.5657,37,"Uruguay has a middle income economy, mainly do..."
3,4,0.5468,14,"As a province of the Viceroyalty of La Plata, ..."
4,5,0.5439,18,Uruguay's politics takes place in a framework ...
